# Notebook 00: Exploring the Synthetic Cohort

This notebook walks through the synthetic patient data generated by
`scripts/01_generate_synthetic_data.py`, visualizes example trajectories,
and explores the parameter distributions.

**Run the data generation script first:**
```bash
python scripts/01_generate_synthetic_data.py
```

In [ ]:
import sys
import os

# Ensure the project root is on the path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams["figure.dpi"] = 120
matplotlib.rcParams["font.size"]  = 10

from src.lotka_volterra import LotkaVolterraModel, DEFAULT_PARAMS
from src.psa_model import PSAModel
from src.utils import plot_trajectory, plot_psa, COLORS

DATA_DIR = os.path.join(project_root, "data", "synthetic")
print("Project root:", project_root)
print("Data dir:", DATA_DIR)

## 1. Default Model — Single Simulation

Start with the published Zhang 2017 parameter set and visualize how the
three cell populations evolve under:
- No treatment
- Continuous MTD treatment
- Zhang adaptive therapy

In [ ]:
model     = LotkaVolterraModel()
psa_model = PSAModel()

print(model)
print(f"Initial cells: T+={model.T0_plus:.0f}, Tp={model.T0_prod:.0f}, T-={model.T0_minus:.0f}")
print(f"Growth rates (day⁻¹): r+={model.r_plus:.5f}, rp={model.r_prod:.5f}, r-={model.r_minus:.5f}")

In [ ]:
t_eval = np.linspace(0, 1825, 1826)

# No treatment
sim_none = model.simulate((0, 1825), t_eval, lambda t: 0)

# MTD
sim_mtd  = model.simulate((0, 1825), t_eval, lambda t: 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, sim, title in [
    (axes[0], sim_none, "No Treatment"),
    (axes[1], sim_mtd,  "MTD (continuous abiraterone)"),
]:
    t = sim["t"]
    total0 = sim["total_cells"][0]
    ax.plot(t, sim["T_plus"]  / total0, color=COLORS["T_plus"],  label="T⁺ (sensitive)")
    ax.plot(t, sim["T_prod"]  / total0, color=COLORS["T_prod"],  label="Tᵖ (producing)")
    ax.plot(t, sim["T_minus"] / total0, color=COLORS["T_minus"], label="T⁻ (resistant)")
    ax.plot(t, sim["total_cells"] / total0, "k--", alpha=0.4, label="Total")
    ax.set_title(title)
    ax.set_xlabel("Days")
    ax.set_ylabel("Cells (normalized)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Zhang Adaptive Therapy Protocol

Stop abiraterone when PSA ≤ 50% of baseline, restart when PSA ≥ 100%.

In [ ]:
# Reproduce the day-by-day adaptive simulation
baseline_psa = psa_model.compute_psa({
    "T_plus":  np.array([model.T0_plus]),
    "T_prod":  np.array([model.T0_prod]),
    "T_minus": np.array([model.T0_minus]),
})[0]

stop_thresh    = 0.50 * baseline_psa
restart_thresh = 1.00 * baseline_psa
drug_on = True

y = np.array([model.T0_plus, model.T0_prod, model.T0_minus], dtype=float)
days_list, on_list, psa_list, T_list = [], [], [], []

for day in range(1826):
    T_plus, T_prod, T_minus = y
    psa_val = psa_model.compute_psa({
        "T_plus":  np.array([T_plus]),
        "T_prod":  np.array([T_prod]),
        "T_minus": np.array([T_minus]),
    })[0]
    psa_norm = psa_val / baseline_psa

    days_list.append(day)
    on_list.append(int(drug_on))
    psa_list.append(psa_norm)
    T_list.append([T_plus, T_prod, T_minus])

    if drug_on and psa_val <= stop_thresh:
        drug_on = False
    elif not drug_on and psa_val >= restart_thresh:
        drug_on = True

    if day < 1825:
        dval = 1 if drug_on else 0
        sub  = model.simulate(
            t_span=(float(day), float(day + 1)),
            t_eval=np.array([float(day + 1)]),
            drug_schedule=lambda t, d=dval: d,
        )
        y = np.array([sub["T_plus"][-1], sub["T_prod"][-1], sub["T_minus"][-1]])

days_arr = np.array(days_list)
on_arr   = np.array(on_list)
psa_arr  = np.array(psa_list)
T_arr    = np.array(T_list)

print(f"% time off treatment: {(on_arr == 0).mean() * 100:.1f}%")

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

# PSA
ax1.plot(days_arr, psa_arr, color=COLORS["psa"], lw=1.5)
ax1.axhline(1.0, color="gray", ls=":", lw=1, label="Baseline")
ax1.axhline(0.5, color="green", ls="--", lw=1, alpha=0.7, label="Stop threshold (50%)")
ax1.axhline(2.0, color="red",  ls="--", lw=1, alpha=0.7, label="Progression (200%)")
ax1.set_ylabel("PSA (normalized)")
ax1.legend(fontsize=8, loc="upper right")
ax1.set_title("Zhang Adaptive Therapy — Nominal Parameters")
ax1.grid(True, alpha=0.3)

# Cell populations
total0 = T_arr[0].sum()
ax2.stackplot(days_arr,
              T_arr[:, 0] / total0,
              T_arr[:, 1] / total0,
              T_arr[:, 2] / total0,
              labels=["T⁺", "Tᵖ", "T⁻"],
              colors=[COLORS["T_plus"], COLORS["T_prod"], COLORS["T_minus"]],
              alpha=0.7)
ax2.set_ylabel("Cell fraction")
ax2.legend(loc="upper left", fontsize=8)
ax2.grid(True, alpha=0.3)

# Drug schedule
ax3.fill_between(days_arr, on_arr, step="post", color=COLORS["drug"], alpha=0.7)
ax3.set_yticks([0, 1])
ax3.set_yticklabels(["Off", "On"])
ax3.set_xlabel("Days")
ax3.set_ylabel("Abiraterone")
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Population Parameter Distributions

Load the synthetic cohort and inspect the sampled parameter distributions.

In [ ]:
import os

params_path = os.path.join(DATA_DIR, "population_params.csv")
if not os.path.exists(params_path):
    print("Run scripts/01_generate_synthetic_data.py first to generate data.")
else:
    pop = pd.read_csv(params_path)
    print(f"Cohort size: {len(pop)} patients")
    display(pop[["r_plus", "r_prod", "r_minus", "delta_plus", "T0_plus", "T0_prod", "T0_minus"]].describe().round(5))

In [ ]:
if os.path.exists(params_path):
    pop = pd.read_csv(params_path)
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 6))
    plot_params = [
        ("r_plus",    "r⁺ (day⁻¹)"),
        ("r_prod",    "rᵖ (day⁻¹)"),
        ("r_minus",   "r⁻ (day⁻¹)"),
        ("delta_plus", "δ⁺ (day⁻¹)"),
        ("T0_plus",   "T⁺(0)"),
        ("T0_prod",   "Tᵖ(0)"),
        ("T0_minus",  "T⁻(0)"),
    ]
    axes_flat = axes.flatten()
    for ax, (col, label) in zip(axes_flat, plot_params):
        ax.hist(pop[col], bins=15, edgecolor="black", alpha=0.7)
        ax.set_title(label)
        ax.set_xlabel(col)
        ax.grid(True, alpha=0.3)
    axes_flat[-1].set_visible(False)  # Hide last empty subplot
    fig.suptitle("Synthetic Cohort — Parameter Distributions (N=50)", y=1.02)
    plt.tight_layout()
    plt.show()

## 4. Sample Trajectories from Synthetic Cohort

In [ ]:
if os.path.exists(params_path):
    fig, axes = plt.subplots(3, 2, figsize=(14, 10))

    for idx, ax_pair in enumerate(axes):
        csv_path = os.path.join(DATA_DIR, f"patient_{idx:03d}_adaptive.csv")
        if not os.path.exists(csv_path):
            continue
        df = pd.read_csv(csv_path)

        ax_psa, ax_drug = ax_pair
        ax_psa.plot(df["day"], df["psa_normalized"], color=COLORS["psa"], lw=1.5)
        ax_psa.axhline(1.0, color="gray", ls=":", lw=1)
        ax_psa.axhline(2.0, color="red",  ls="--", lw=1, alpha=0.5)
        ax_psa.set_title(f"Patient {idx:03d} — PSA")
        ax_psa.set_ylabel("PSA (normalized)")
        ax_psa.grid(True, alpha=0.3)

        ax_drug.fill_between(df["day"], df["on_treatment"], step="post",
                             color=COLORS["drug"], alpha=0.6)
        ax_drug.set_yticks([0, 1])
        ax_drug.set_yticklabels(["Off", "On"])
        ax_drug.set_title(f"Patient {idx:03d} — Drug Schedule")
        ax_drug.grid(True, alpha=0.3)

    axes[-1, 0].set_xlabel("Days")
    axes[-1, 1].set_xlabel("Days")
    fig.suptitle("Sample Trajectories — Zhang Adaptive Therapy (Patients 0–2)", y=1.02)
    plt.tight_layout()
    plt.show()

## 5. Protocol Comparison Results

Load the protocol comparison output from script 03 (if available).

In [ ]:
comparison_path = os.path.join(DATA_DIR, "protocol_comparison.csv")
if not os.path.exists(comparison_path):
    print("Run scripts/03_simulate_treatments.py first.")
else:
    df_comp = pd.read_csv(comparison_path)
    summary = df_comp.groupby("protocol").agg(
        mean_ttp=("ttp", lambda x: np.mean(np.where(np.isinf(x), 1825, x))),
        median_ttp=("ttp", lambda x: np.median(np.where(np.isinf(x), 1825, x))),
        mean_drug_days=("drug_days", "mean"),
        pct_progressed=("ttp", lambda x: np.mean(~np.isinf(x)) * 100),
    ).round(1)
    print(summary.to_string())

In [ ]:
figures_dir = os.path.join(DATA_DIR, "figures")
if os.path.exists(figures_dir):
    from IPython.display import Image, display as ipy_display
    for fname in ["km_curves.png", "ttp_scatter.png", "dose_boxplot.png"]:
        fpath = os.path.join(figures_dir, fname)
        if os.path.exists(fpath):
            print(f"\n--- {fname} ---")
            ipy_display(Image(fpath))

## 6. Sensitivity: Effect of delta_plus on TTP

Sweep the drug effect parameter `delta_plus` to see how TTP under MTD depends on drug efficacy.

In [ ]:
delta_values = np.linspace(0.02, 0.50, 25)
ttp_vs_delta = []

for delta in delta_values:
    params = {**DEFAULT_PARAMS, "delta_plus": delta, "delta_prod": delta / 2}
    m = LotkaVolterraModel(params)
    t_eval = np.linspace(0, 1825, 1826)
    sim = m.simulate((0, 1825), t_eval, lambda t: 1)
    ttp = m.time_to_progression(sim, progression_threshold=2.0)
    ttp_vs_delta.append(min(ttp, 1825))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(delta_values, ttp_vs_delta, "o-", color="steelblue", markersize=5)
ax.axvline(DEFAULT_PARAMS["delta_plus"], color="red", ls="--",
           label=f"Default δ⁺={DEFAULT_PARAMS['delta_plus']}")
ax.set_xlabel("δ⁺ (abiraterone kill rate, day⁻¹)")
ax.set_ylabel("TTP under MTD (days, capped at 1825)")
ax.set_title("Sensitivity of TTP to Drug Efficacy (MTD)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Phase Portrait: T+ vs T- under Different Protocols

Visualize the evolutionary trajectory in the (T+, T-) plane.

In [ ]:
model_phase = LotkaVolterraModel()
t_eval_ph   = np.linspace(0, 1825, 1826)

sim_mtd_ph  = model_phase.simulate((0, 1825), t_eval_ph, lambda t: 1)
sim_none_ph = model_phase.simulate((0, 1825), t_eval_ph, lambda t: 0)

fig, ax = plt.subplots(figsize=(7, 6))

for sim, label, color in [
    (sim_mtd_ph,  "MTD",          "#e377c2"),
    (sim_none_ph, "No treatment", "steelblue"),
]:
    T_plus  = sim["T_plus"]
    T_minus = sim["T_minus"]
    sc = ax.scatter(T_plus, T_minus, c=sim["t"], cmap="plasma",
                    s=6, alpha=0.8, label=label)
    ax.plot(T_plus[0],  T_minus[0],  "*", color=color, ms=12, markeredgecolor="k")
    ax.plot(T_plus[-1], T_minus[-1], "^", color=color, ms=10, markeredgecolor="k")

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("Time (days)")
ax.set_xlabel("T⁺ (sensitive cells)")
ax.set_ylabel("T⁻ (resistant cells)")
ax.set_title("Phase Portrait: T⁺ vs T⁻  (★ = start, ▲ = end)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## Next Steps

- **Phase 2**: Run `scripts/02_fit_patient_parameters.py` to fit the model to each synthetic patient and quantify parameter uncertainty via MCMC.
- **Phase 3**: Use the fitted uncertainty sets as input to a convex robust optimization over treatment thresholds.
- **Phase 4**: Implement Wasserstein distributionally robust treatment policies that minimize worst-case regret over the inferred parameter distribution.

See `README.md` for the full project roadmap.